*0.1 Python for GenAI*

# uv

**The situation.** CI is green. Staging is green. Production breaks. Nobody changed a line of code. Between the two deploys, a library your library depends on released a new version, and `pip install` on each machine pulled whatever was newest that day. "It works on my machine" — because your machine installed it last week.

**The fix: record exactly what was installed, and install exactly that everywhere.** `uv` keeps two files. `pyproject.toml` says what you asked for (`httpx>=0.27`). `uv.lock` records what was actually resolved — every package, every version, including the dependencies of dependencies. Every machine installs from the lock and gets the identical set.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**A new service, from scratch.** Create a project, add two libraries and two development tools, and look at what uv wrote.

In [2]:
import subprocess
import tempfile
import tomllib
from pathlib import Path

folder = tempfile.mkdtemp()
project = Path(folder) / "support-bot"
subprocess.run(
    ["uv", "init", "--no-workspace", "--python", "3.12", str(project)],
    capture_output=True,
    check=True,
)
subprocess.run(["uv", "add", "httpx", "pydantic"], cwd=project, capture_output=True, check=True)
subprocess.run(
    ["uv", "add", "--dev", "pytest", "ruff"], cwd=project, capture_output=True, check=True
)

pyproject = tomllib.loads((project / "pyproject.toml").read_text())
lock = tomllib.loads((project / "uv.lock").read_text())
print("you asked for:     ", pyproject["project"]["dependencies"])
print("dev tools:         ", pyproject["dependency-groups"]["dev"])
print(
    "uv.lock pinned:    ",
    len(lock["package"]),
    "packages, e.g.",
    lock["package"][0]["name"],
    lock["package"][0]["version"],
)
assert len(lock["package"]) > 10

you asked for:      ['httpx>=0.28.1', 'pydantic>=2.13.5']
dev tools:          ['pytest>=9.1.1', 'ruff>=0.16.8']
uv.lock pinned:     19 packages, e.g. annotated-types 0.8.0


**Reading the output.** You asked for 2 libraries; the lock pinned many more — everything those 2 need, each at one exact version. That file is what CI and production install from.

**Prove the environment is real.** Run Python inside the project's own environment.

In [3]:
run = subprocess.run(
    ["uv", "run", "python", "-c", "import httpx, sys; print(httpx.__version__, sys.prefix)"],
    cwd=project,
    capture_output=True,
    text=True,
    check=True,
)
version, prefix = run.stdout.split()
print("httpx", version, "installed in", prefix.replace(folder, "…"))
print("CI and Docker install with: uv sync --frozen   (fails loudly if the lock is out of date)")
assert prefix.endswith(".venv")

httpx 0.28.1 installed in /private…/support-bot/.venv
CI and Docker install with: uv sync --frozen   (fails loudly if the lock is out of date)


**The rule to remember.** Commit `uv.lock`. Install with `uv sync --frozen` everywhere. Upgrade on purpose (`uv lock --upgrade`, run the tests), never by accident.

```
pyproject.toml   httpx>=0.27          what you want
     │  uv lock
     ▼
uv.lock          httpx==0.28.1 +23    what was resolved  ──uv sync --frozen──▶  laptop · CI · Docker: identical
```

| Use it when | Don't when | Instead use |
|---|---|---|
| new Python projects | the team already uses Poetry — one tool per repo | Poetry (next item) |

**Watch out**
- `pip install` inside a uv project is invisible to the lock; the next `uv sync` removes it and someone wonders why.
- Pin the Python version too (`.python-version`); a different interpreter is a dependency change.
- In Docker, copy `pyproject.toml` and `uv.lock` and sync *before* copying source, so the dependency layer is cached across code changes.